# Klasifikasi Bunga: Random Forest + MobileNetV2
## Mata Kuliah Machine Learning — UAS

**Dataset:** Bunga Melati Jakarta, Melati Jepang, Bintaro, dan Tapak Dara  
**Referensi:** Koklu et al. (2022) — *A CNN-SVM study based on selected deep features for grapevine leaves classification*, Measurement 188, 110425

---
### Alur Penelitian:
1. Load & Preprocessing Dataset
2. Data Augmentation
3. **Metode ML (UTS):** Random Forest
4. **Metode Deep Learning (UAS):** MobileNetV2 (Fine-Tuning + Feature Extraction)
5. Evaluasi & Perbandingan Model
6. Visualisasi Hasil

## 1. Import Library

In [ ]:
# ============================================================
# IMPORT LIBRARY
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Image Processing
from PIL import Image
import cv2

# Sklearn (Random Forest - Metode UTS)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    matthews_corrcoef
)
from sklearn.feature_selection import chi2, SelectKBest

# TensorFlow / Keras (MobileNetV2 - Metode DL UAS)
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices('GPU'))
print("Library berhasil diimpor!")

## 2. Konfigurasi & Struktur Dataset

Pastikan struktur folder dataset Anda seperti berikut:
```
dataset/
├── melati_jakarta/     (360 gambar)
├── melati_jepang/      (360 gambar)
├── bintaro/            (360 gambar)
└── tapak_dara/         (360 gambar)
```

In [ ]:
# ============================================================
# KONFIGURASI PARAMETER
# ============================================================

# ⚠️ SESUAIKAN PATH DATASET ANDA
DATASET_PATH = './dataset'  # Ganti dengan path dataset Anda

# Kelas bunga
CLASS_NAMES = ['melati_jakarta', 'melati_jepang', 'bintaro', 'tapak_dara']
NUM_CLASSES = len(CLASS_NAMES)

# Parameter gambar
IMG_SIZE = 224  # MobileNetV2 input size
IMG_SIZE_RF = 64  # Ukuran untuk Random Forest (lebih kecil agar cepat)

# Parameter training
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001
TEST_SIZE = 0.2   # 80% train, 20% test
VAL_SIZE = 0.1    # 10% dari train untuk validasi
RANDOM_STATE = 42

# Parameter Random Forest
RF_N_ESTIMATORS = 100

print("Konfigurasi:")
print(f"  Jumlah kelas     : {NUM_CLASSES}")
print(f"  Kelas            : {CLASS_NAMES}")
print(f"  Ukuran gambar DL : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Ukuran gambar RF : {IMG_SIZE_RF}x{IMG_SIZE_RF}")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Epochs           : {EPOCHS}")
print(f"  Split test       : {int(TEST_SIZE*100)}%")

## 3. Load & Preprocessing Dataset

In [ ]:
# ============================================================
# FUNGSI LOAD DATASET
# ============================================================

def load_dataset(dataset_path, class_names, img_size):
    """
    Load gambar dari folder dataset.
    
    Args:
        dataset_path : path ke folder dataset
        class_names  : list nama kelas/folder
        img_size     : ukuran resize gambar
    
    Returns:
        X (array gambar), y (array label)
    """
    X, y = [], []
    
    for label, class_name in enumerate(class_names):
        class_path = os.path.join(dataset_path, class_name)
        
        if not os.path.exists(class_path):
            print(f"⚠️  Folder tidak ditemukan: {class_path}")
            continue
        
        images = [f for f in os.listdir(class_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
        
        print(f"  [{class_name}] → {len(images)} gambar ditemukan")
        
        for img_file in images:
            img_path = os.path.join(class_path, img_file)
            try:
                img = Image.open(img_path).convert('RGB')
                img = img.resize((img_size, img_size))
                img_array = np.array(img)
                X.append(img_array)
                y.append(label)
            except Exception as e:
                print(f"  ⚠️  Gagal load: {img_file} — {e}")
    
    return np.array(X), np.array(y)


# Load dataset untuk DL (224x224)
print("\nLoading dataset untuk MobileNetV2 (224x224)...")
X_dl, y = load_dataset(DATASET_PATH, CLASS_NAMES, IMG_SIZE)

print(f"\nTotal gambar loaded : {len(X_dl)}")
print(f"Shape X             : {X_dl.shape}")
print(f"Shape y             : {y.shape}")
print(f"Distribusi kelas    : {dict(zip(CLASS_NAMES, [np.sum(y==i) for i in range(NUM_CLASSES)]))}")

In [ ]:
# ============================================================
# VISUALISASI SAMPEL DATASET
# ============================================================

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Sampel Dataset Bunga', fontsize=16, fontweight='bold')

for i, class_name in enumerate(CLASS_NAMES):
    class_indices = np.where(y == i)[0]
    sample_indices = np.random.choice(class_indices, 5, replace=False)
    
    for j, idx in enumerate(sample_indices):
        axes[i, j].imshow(X_dl[idx])
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(class_name.replace('_', ' ').title(),
                                  fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('sampel_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gambar sampel dataset disimpan: sampel_dataset.png")

In [ ]:
# ============================================================
# PREPROCESSING DATA
# ============================================================

# --- Preprocessing untuk MobileNetV2 (normalisasi [-1, 1]) ---
X_dl_norm = preprocess_input(X_dl.astype(np.float32))  # MobileNetV2 preprocessing

# --- Preprocessing untuk Random Forest (flatten + normalisasi [0, 1]) ---
X_rf = X_dl.astype(np.float32) / 255.0  # Normalisasi 0-1
X_rf_flat = X_rf.reshape(len(X_rf), -1)  # Flatten: (N, 224*224*3)

# Karena 224x224x3 terlalu besar untuk RF, gunakan resize lebih kecil
print("Resizing gambar untuk Random Forest (64x64)...")
X_rf_small = []
for img in X_dl:
    img_small = cv2.resize(img, (IMG_SIZE_RF, IMG_SIZE_RF))
    X_rf_small.append(img_small)
X_rf_small = np.array(X_rf_small).astype(np.float32) / 255.0
X_rf_flat = X_rf_small.reshape(len(X_rf_small), -1)

# One-hot encoding untuk DL
y_onehot = to_categorical(y, NUM_CLASSES)

print(f"Shape X_dl_norm   : {X_dl_norm.shape}  ← untuk MobileNetV2")
print(f"Shape X_rf_flat   : {X_rf_flat.shape} ← untuk Random Forest")
print(f"Shape y_onehot    : {y_onehot.shape}")

In [ ]:
# ============================================================
# PEMBAGIAN DATA TRAIN & TEST
# ============================================================

# Split untuk Deep Learning
X_train_dl, X_test_dl, y_train_oh, y_test_oh, y_train, y_test = train_test_split(
    X_dl_norm, y_onehot, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Split validasi dari data training (untuk DL)
X_train_dl, X_val_dl, y_train_oh, y_val_oh = train_test_split(
    X_train_dl, y_train_oh,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_STATE
)

# Split untuk Random Forest
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf_flat, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Pembagian Dataset:")
print(f"  Total data          : {len(X_dl_norm)}")
print(f"  Train DL            : {len(X_train_dl)} ({len(X_train_dl)/len(X_dl_norm)*100:.1f}%)")
print(f"  Validasi DL         : {len(X_val_dl)} ({len(X_val_dl)/len(X_dl_norm)*100:.1f}%)")
print(f"  Test                : {len(X_test_dl)} ({len(X_test_dl)/len(X_dl_norm)*100:.1f}%)")
print(f"  Train RF            : {len(X_train_rf)}")
print(f"  Test RF             : {len(X_test_rf)}")

## 4. Data Augmentation

Mengacu pada Koklu et al. (2022): augmentasi dilakukan dengan teknik refleksi, rotasi, scaling, dan translasi.

In [ ]:
# ============================================================
# DATA AUGMENTATION (untuk MobileNetV2)
# Referensi: Koklu et al. (2022) - Tabel 2
# ============================================================

train_datagen = ImageDataGenerator(
    horizontal_flip=True,        # Refleksi horizontal
    vertical_flip=False,
    rotation_range=45,           # Rotasi ±45° (sesuai jurnal)
    zoom_range=0.2,              # Scale 80%-120% (sesuai jurnal)
    width_shift_range=0.1,       # Translasi horizontal
    height_shift_range=0.1,      # Translasi vertikal
    brightness_range=[0.8, 1.2], # Variasi kecerahan
    fill_mode='nearest'
)

# Tidak ada augmentasi untuk validasi dan test
val_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

# Buat generator
train_generator = train_datagen.flow(
    X_train_dl, y_train_oh,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_generator = val_datagen.flow(
    X_val_dl, y_val_oh,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Visualisasi efek augmentasi
sample_img = X_dl[0]
sample_img_batch = sample_img[np.newaxis, ...]  # tambah dimensi batch
aug_gen = train_datagen.flow(sample_img_batch, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Efek Data Augmentation', fontsize=14, fontweight='bold')
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original', fontsize=9)
axes[0, 0].axis('off')

for idx in range(1, 10):
    aug_img = next(aug_gen)[0].astype(np.uint8)
    row, col = idx // 5, idx % 5
    axes[row, col].imshow(aug_img)
    axes[row, col].set_title(f'Augmented {idx}', fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('data_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gambar augmentasi disimpan: data_augmentation.png")

## 5. Metode Machine Learning (UTS) — Random Forest

Metode Random Forest digunakan sebagai baseline (metode dari UTS). Fitur berupa piksel gambar yang di-flatten.

In [ ]:
# ============================================================
# METODE ML (UTS): RANDOM FOREST
# ============================================================

print("=" * 60)
print("METODE ML (UTS): RANDOM FOREST")
print("=" * 60)

# Inisialisasi model Random Forest
rf_model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,  # 100 pohon keputusan
    max_depth=None,                 # Kedalaman tidak dibatasi
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',            # Fitur = sqrt(total fitur)
    n_jobs=-1,                      # Gunakan semua CPU
    random_state=RANDOM_STATE,
    verbose=1
)

# Training
print(f"\nTraining Random Forest ({RF_N_ESTIMATORS} estimators)...")
rf_model.fit(X_train_rf, y_train_rf)

# Prediksi
y_pred_rf = rf_model.predict(X_test_rf)

# Evaluasi
rf_acc  = accuracy_score(y_test_rf, y_pred_rf)
rf_prec = precision_score(y_test_rf, y_pred_rf, average='weighted')
rf_rec  = recall_score(y_test_rf, y_pred_rf, average='weighted')
rf_f1   = f1_score(y_test_rf, y_pred_rf, average='weighted')
rf_mcc  = matthews_corrcoef(y_test_rf, y_pred_rf)

print(f"\n{'='*45}")
print(f"  HASIL RANDOM FOREST")
print(f"{'='*45}")
print(f"  Accuracy   : {rf_acc*100:.2f}%")
print(f"  Precision  : {rf_prec:.4f}")
print(f"  Recall     : {rf_rec:.4f}")
print(f"  F1-Score   : {rf_f1:.4f}")
print(f"  MCC        : {rf_mcc:.4f}")
print(f"{'='*45}")

# Simpan metrik
rf_metrics = {
    'Model': 'Random Forest (ML - UTS)',
    'Accuracy (%)': round(rf_acc * 100, 2),
    'Precision': round(rf_prec, 4),
    'Recall (Sensitivity)': round(rf_rec, 4),
    'F1-Score': round(rf_f1, 4),
    'MCC': round(rf_mcc, 4)
}

In [ ]:
# ============================================================
# VISUALISASI CONFUSION MATRIX — RANDOM FOREST
# ============================================================

cm_rf = confusion_matrix(y_test_rf, y_pred_rf)
class_labels = [c.replace('_', ' ').title() for c in CLASS_NAMES]

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_rf, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_labels, yticklabels=class_labels
)
plt.title(f'Confusion Matrix — Random Forest\nAccuracy: {rf_acc*100:.2f}%',
          fontsize=13, fontweight='bold')
plt.ylabel('Aktual', fontsize=11)
plt.xlabel('Prediksi', fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('cm_random_forest.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix RF disimpan: cm_random_forest.png")

# Classification Report
print("\nClassification Report — Random Forest:")
print(classification_report(y_test_rf, y_pred_rf, target_names=class_labels))

In [ ]:
# ============================================================
# FEATURE IMPORTANCE — RANDOM FOREST
# ============================================================

importances = rf_model.feature_importances_
# Reshape ke ukuran gambar untuk visualisasi
importance_map = importances.reshape(IMG_SIZE_RF, IMG_SIZE_RF, 3)
importance_gray = importance_map.mean(axis=2)  # Rata-rata 3 channel

plt.figure(figsize=(7, 5))
plt.imshow(importance_gray, cmap='hot')
plt.colorbar(label='Importance')
plt.title('Feature Importance Map — Random Forest\n(Merah = Lebih Penting)',
          fontsize=12, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print("Feature importance disimpan: feature_importance_rf.png")

## 6. Metode Deep Learning (UAS) — MobileNetV2

Mengacu pada Koklu et al. (2022), digunakan tiga pendekatan:
1. **Fine-tuning MobileNetV2** — klasifikasi langsung
2. **MobileNetV2 + Random Forest** (CNN sebagai feature extractor, RF sebagai classifier)
3. **MobileNetV2 + Feature Selection + Random Forest** (Chi-Square, sesuai jurnal)

In [ ]:
# ============================================================
# DL METHOD 1: FINE-TUNING MobileNetV2
# ============================================================

print("=" * 60)
print("DEEP LEARNING: FINE-TUNING MobileNetV2")
print("=" * 60)

# Load MobileNetV2 pre-trained (ImageNet), tanpa top layer
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze semua layer base model dulu
base_model.trainable = False

# Tambah layer klasifikasi
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)  # Output 4 kelas

mobilenet_model = Model(inputs=base_model.input, outputs=outputs)

# Kompilasi
mobilenet_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mobilenet_model.summary()

In [ ]:
# ============================================================
# TRAINING MOBILENETV2 — PHASE 1 (Feature Extraction)
# ============================================================

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

steps_per_epoch = len(X_train_dl) // BATCH_SIZE

print(f"Training Phase 1 (base frozen) — {EPOCHS} epochs...")
history1 = mobilenet_model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ============================================================
# TRAINING MOBILENETV2 — PHASE 2 (Fine-Tuning)
# Unfreeze beberapa layer terakhir untuk fine-tuning
# ============================================================

print("Fine-tuning: membuka 50 layer terakhir...")
base_model.trainable = True

# Freeze semua kecuali 50 layer terakhir
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Kompilasi ulang dengan learning rate lebih kecil
mobilenet_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Training Phase 2 (fine-tuning) — 10 epochs...")
history2 = mobilenet_model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining selesai!")

In [ ]:
# ============================================================
# VISUALISASI GRAFIK AKURASI & LOSS — MobileNetV2
# ============================================================

# Gabungkan history phase 1 dan 2
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history.get(key, [])
    return combined

all_history = combine_histories(history1, history2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History — MobileNetV2', fontsize=14, fontweight='bold')

# Akurasi
axes[0].plot(all_history['accuracy'], label='Training', color='blue', linewidth=2)
axes[0].plot(all_history['val_accuracy'], label='Validasi', color='orange',
             linestyle='--', linewidth=2)
axes[0].axvline(x=len(history1.history['accuracy'])-1, color='red',
                linestyle=':', alpha=0.7, label='Fine-tuning mulai')
axes[0].set_title('Grafik Akurasi', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(all_history['loss'], label='Training', color='blue', linewidth=2)
axes[1].plot(all_history['val_loss'], label='Validasi', color='orange',
             linestyle='--', linewidth=2)
axes[1].axvline(x=len(history1.history['loss'])-1, color='red',
                linestyle=':', alpha=0.7, label='Fine-tuning mulai')
axes[1].set_title('Grafik Loss', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grafik_training_mobilenetv2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafik training disimpan: grafik_training_mobilenetv2.png")

In [ ]:
# ============================================================
# EVALUASI MobileNetV2 (Fine-Tuning)
# ============================================================

y_pred_dl_prob = mobilenet_model.predict(X_test_dl, verbose=0)
y_pred_dl = np.argmax(y_pred_dl_prob, axis=1)

dl_acc  = accuracy_score(y_test, y_pred_dl)
dl_prec = precision_score(y_test, y_pred_dl, average='weighted')
dl_rec  = recall_score(y_test, y_pred_dl, average='weighted')
dl_f1   = f1_score(y_test, y_pred_dl, average='weighted')
dl_mcc  = matthews_corrcoef(y_test, y_pred_dl)

print(f"\n{'='*45}")
print(f"  HASIL MobileNetV2 (Fine-Tuning)")
print(f"{'='*45}")
print(f"  Accuracy   : {dl_acc*100:.2f}%")
print(f"  Precision  : {dl_prec:.4f}")
print(f"  Recall     : {dl_rec:.4f}")
print(f"  F1-Score   : {dl_f1:.4f}")
print(f"  MCC        : {dl_mcc:.4f}")
print(f"{'='*45}")

dl_metrics = {
    'Model': 'MobileNetV2 Fine-Tuning (DL)',
    'Accuracy (%)': round(dl_acc * 100, 2),
    'Precision': round(dl_prec, 4),
    'Recall (Sensitivity)': round(dl_rec, 4),
    'F1-Score': round(dl_f1, 4),
    'MCC': round(dl_mcc, 4)
}

# Confusion Matrix
cm_dl = confusion_matrix(y_test, y_pred_dl)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_dl, annot=True, fmt='d', cmap='Greens',
    xticklabels=class_labels, yticklabels=class_labels
)
plt.title(f'Confusion Matrix — MobileNetV2 Fine-Tuning\nAccuracy: {dl_acc*100:.2f}%',
          fontsize=13, fontweight='bold')
plt.ylabel('Aktual', fontsize=11)
plt.xlabel('Prediksi', fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('cm_mobilenetv2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix MobileNetV2 disimpan: cm_mobilenetv2.png")

## 7. CNN-RF Hybrid: MobileNetV2 sebagai Feature Extractor + Random Forest

Mengacu pada Koklu et al. (2022): fitur dari layer Logits MobileNetV2 diekstrak, lalu diklasifikasikan dengan RF.

In [ ]:
# ============================================================
# EKSTRAKSI DEEP FEATURES dari MobileNetV2
# Referensi: Koklu et al. (2022) - Section 2.4
# ============================================================

print("=" * 60)
print("EKSTRAKSI DEEP FEATURES — MobileNetV2 Logits Layer")
print("=" * 60)

# Buat model feature extractor (ambil dari GlobalAveragePooling2D)
feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.get_layer('global_average_pooling2d' if 'global_average_pooling2d'
                                  in [l.name for l in base_model.layers]
                                  else base_model.layers[-1].name).output
)

# Gunakan base model output langsung untuk feature extraction
# (setelah GlobalAveragePooling2D dari full model)
feature_layer_name = None
for layer in mobilenet_model.layers:
    if isinstance(layer, GlobalAveragePooling2D):
        feature_layer_name = layer.name
        break

feature_extractor_model = Model(
    inputs=mobilenet_model.input,
    outputs=mobilenet_model.get_layer(feature_layer_name).output
)

# Gabungkan semua data untuk ekstraksi
X_all_norm = np.vstack([X_train_dl, X_val_dl, X_test_dl])
y_all = np.concatenate([
    np.argmax(y_train_oh, axis=1),
    np.argmax(y_val_oh, axis=1),
    y_test
])

print(f"Mengekstrak fitur dari {len(X_all_norm)} gambar...")
features_all = feature_extractor_model.predict(X_all_norm, batch_size=32, verbose=1)
print(f"Shape fitur yang diekstrak: {features_all.shape}")

# Split kembali ke train/test
n_train = len(X_train_dl) + len(X_val_dl)
features_train = features_all[:n_train]
features_test  = features_all[n_train:]
y_feat_train   = y_all[:n_train]
y_feat_test    = y_all[n_train:]

print(f"Fitur train : {features_train.shape}")
print(f"Fitur test  : {features_test.shape}")

In [ ]:
# ============================================================
# DL METHOD 2: CNN + RANDOM FOREST (Tanpa Feature Selection)
# ============================================================

print("=" * 60)
print("CNN-RF: MobileNetV2 Features + Random Forest")
print("=" * 60)

rf_cnn = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_cnn.fit(features_train, y_feat_train)

y_pred_cnn_rf = rf_cnn.predict(features_test)

cnnrf_acc  = accuracy_score(y_feat_test, y_pred_cnn_rf)
cnnrf_prec = precision_score(y_feat_test, y_pred_cnn_rf, average='weighted')
cnnrf_rec  = recall_score(y_feat_test, y_pred_cnn_rf, average='weighted')
cnnrf_f1   = f1_score(y_feat_test, y_pred_cnn_rf, average='weighted')
cnnrf_mcc  = matthews_corrcoef(y_feat_test, y_pred_cnn_rf)

print(f"\n{'='*45}")
print(f"  HASIL CNN + Random Forest")
print(f"{'='*45}")
print(f"  Accuracy   : {cnnrf_acc*100:.2f}%")
print(f"  Precision  : {cnnrf_prec:.4f}")
print(f"  Recall     : {cnnrf_rec:.4f}")
print(f"  F1-Score   : {cnnrf_f1:.4f}")
print(f"  MCC        : {cnnrf_mcc:.4f}")
print(f"{'='*45}")

cnnrf_metrics = {
    'Model': 'CNN + Random Forest (semua fitur)',
    'Accuracy (%)': round(cnnrf_acc * 100, 2),
    'Precision': round(cnnrf_prec, 4),
    'Recall (Sensitivity)': round(cnnrf_rec, 4),
    'F1-Score': round(cnnrf_f1, 4),
    'MCC': round(cnnrf_mcc, 4)
}

In [ ]:
# ============================================================
# DL METHOD 3: CNN + FEATURE SELECTION (Chi-Square) + RF
# Referensi: Koklu et al. (2022) - Section 2.4.1
# ============================================================

print("=" * 60)
print("CNN + Chi-Square Feature Selection + Random Forest")
print("=" * 60)

# Normalisasi fitur agar nilai non-negatif (syarat Chi-Square)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
features_train_scaled = scaler.fit_transform(features_train)
features_test_scaled  = scaler.transform(features_test)

# Jumlah fitur awal
n_features_total = features_train_scaled.shape[1]
n_features_selected = n_features_total // 4  # Pilih 25% fitur terbaik (≈ jurnal: 1000→250)

print(f"Jumlah fitur total    : {n_features_total}")
print(f"Jumlah fitur dipilih  : {n_features_selected}")

# Feature selection dengan Chi-Square
selector = SelectKBest(score_func=chi2, k=n_features_selected)
features_train_selected = selector.fit_transform(features_train_scaled, y_feat_train)
features_test_selected  = selector.transform(features_test_scaled)

print(f"Shape setelah seleksi (train): {features_train_selected.shape}")
print(f"Shape setelah seleksi (test) : {features_test_selected.shape}")

# Random Forest dengan fitur terpilih
rf_cnn_fs = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_cnn_fs.fit(features_train_selected, y_feat_train)

y_pred_cnn_rf_fs = rf_cnn_fs.predict(features_test_selected)

cnnrf_fs_acc  = accuracy_score(y_feat_test, y_pred_cnn_rf_fs)
cnnrf_fs_prec = precision_score(y_feat_test, y_pred_cnn_rf_fs, average='weighted')
cnnrf_fs_rec  = recall_score(y_feat_test, y_pred_cnn_rf_fs, average='weighted')
cnnrf_fs_f1   = f1_score(y_feat_test, y_pred_cnn_rf_fs, average='weighted')
cnnrf_fs_mcc  = matthews_corrcoef(y_feat_test, y_pred_cnn_rf_fs)

print(f"\n{'='*45}")
print(f"  HASIL CNN + Chi-Square + Random Forest")
print(f"{'='*45}")
print(f"  Accuracy   : {cnnrf_fs_acc*100:.2f}%")
print(f"  Precision  : {cnnrf_fs_prec:.4f}")
print(f"  Recall     : {cnnrf_fs_rec:.4f}")
print(f"  F1-Score   : {cnnrf_fs_f1:.4f}")
print(f"  MCC        : {cnnrf_fs_mcc:.4f}")
print(f"{'='*45}")

cnnrf_fs_metrics = {
    'Model': 'CNN + Chi-Square + Random Forest (fitur terpilih)',
    'Accuracy (%)': round(cnnrf_fs_acc * 100, 2),
    'Precision': round(cnnrf_fs_prec, 4),
    'Recall (Sensitivity)': round(cnnrf_fs_rec, 4),
    'F1-Score': round(cnnrf_fs_f1, 4),
    'MCC': round(cnnrf_fs_mcc, 4)
}

## 8. Perbandingan Semua Model

In [ ]:
# ============================================================
# TABEL PERBANDINGAN SEMUA MODEL
# ============================================================

all_results = pd.DataFrame([
    rf_metrics,
    dl_metrics,
    cnnrf_metrics,
    cnnrf_fs_metrics
])

print("\n" + "=" * 80)
print("PERBANDINGAN SEMUA MODEL")
print("=" * 80)
print(all_results.to_string(index=False))
print("=" * 80)

# Tampilkan model terbaik
best_idx = all_results['Accuracy (%)'].idxmax()
best_model = all_results.loc[best_idx, 'Model']
best_acc   = all_results.loc[best_idx, 'Accuracy (%)']
print(f"\n✅ MODEL TERBAIK: {best_model}")
print(f"   Accuracy: {best_acc}%")

# Simpan ke CSV
all_results.to_csv('perbandingan_model.csv', index=False)
print("\nTabel perbandingan disimpan: perbandingan_model.csv")

In [ ]:
# ============================================================
# VISUALISASI PERBANDINGAN MODEL
# ============================================================

metrics_to_plot = ['Accuracy (%)', 'Precision', 'Recall (Sensitivity)', 'F1-Score', 'MCC']
model_names = [
    'Random Forest\n(ML - UTS)',
    'MobileNetV2\nFine-Tuning',
    'CNN + RF\n(Semua Fitur)',
    'CNN + Chi²\n+ RF'
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Perbandingan Performa Model', fontsize=15, fontweight='bold')

# Bar chart Accuracy
accs = all_results['Accuracy (%)'].values
colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']
bars = axes[0].bar(model_names, accs, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Akurasi Model (%)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim([max(0, min(accs) - 10), 100])
axes[0].grid(True, axis='y', alpha=0.3)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

# Radar/Grouped bar chart: semua metrik
x = np.arange(len(model_names))
width = 0.15
metric_colors = ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6', '#F39C12']
plot_metrics = ['Precision', 'Recall (Sensitivity)', 'F1-Score', 'MCC']
for i, (metric, color) in enumerate(zip(plot_metrics, metric_colors)):
    vals = all_results[metric].values
    axes[1].bar(x + i * width, vals, width, label=metric, color=color,
                edgecolor='black', linewidth=0.5, alpha=0.85)

axes[1].set_title('Metrik Evaluasi per Model', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(model_names, fontsize=8)
axes[1].legend(fontsize=8)
axes[1].set_ylim([0, 1.1])
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafik perbandingan disimpan: perbandingan_model.png")

In [ ]:
# ============================================================
# CONFUSION MATRIX SEMUA MODEL (PANEL)
# ============================================================

all_preds = [y_pred_rf, y_pred_dl, y_pred_cnn_rf, y_pred_cnn_rf_fs]
all_trues = [y_test_rf, y_test, y_feat_test, y_feat_test]
all_accs  = [rf_acc, dl_acc, cnnrf_acc, cnnrf_fs_acc]
all_titles = [
    'Random Forest (ML-UTS)',
    'MobileNetV2 Fine-Tuning',
    'CNN + RF (All Features)',
    'CNN + Chi² + RF'
]
cmaps = ['Blues', 'Greens', 'Oranges', 'Purples']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Confusion Matrix — Semua Model', fontsize=15, fontweight='bold')

for ax, preds, trues, acc, title, cmap in zip(
        axes.flat, all_preds, all_trues, all_accs, all_titles, cmaps):
    cm = confusion_matrix(trues, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_title(f'{title}\nAcc: {acc*100:.2f}%', fontsize=11, fontweight='bold')
    ax.set_ylabel('Aktual')
    ax.set_xlabel('Prediksi')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('confusion_matrix_semua_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix semua model disimpan: confusion_matrix_semua_model.png")

## 9. Demo Prediksi dengan Input Baru

In [ ]:
# ============================================================
# DEMO PREDIKSI DENGAN INPUT BARU
# ============================================================

def predict_flower(image_path, model_dl, feature_extractor, rf_model_fs,
                   scaler, selector, class_names, img_size=224):
    """
    Fungsi prediksi untuk gambar baru.
    Menampilkan hasil prediksi dari semua model.
    
    Args:
        image_path : path ke gambar yang ingin diprediksi
    """
    # Load dan preprocessing gambar
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((img_size, img_size))
    img_array = np.array(img_resized)
    img_preprocessed = preprocess_input(img_array.astype(np.float32))
    img_batch = np.expand_dims(img_preprocessed, axis=0)
    
    # Prediksi MobileNetV2
    pred_dl_prob = model_dl.predict(img_batch, verbose=0)[0]
    pred_dl_class = np.argmax(pred_dl_prob)
    
    # Ekstraksi fitur untuk CNN+RF
    deep_feat = feature_extractor.predict(img_batch, verbose=0)
    deep_feat_scaled = scaler.transform(deep_feat)
    deep_feat_selected = selector.transform(deep_feat_scaled)
    pred_cnnrf_class = rf_model_fs.predict(deep_feat_selected)[0]
    pred_cnnrf_prob = rf_model_fs.predict_proba(deep_feat_selected)[0]
    
    # Tampilkan hasil
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Gambar input
    axes[0].imshow(np.array(img_resized))
    axes[0].set_title('Input Gambar', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Prediksi MobileNetV2
    label_names = [c.replace('_', ' ').title() for c in class_names]
    colors_bar = ['#E74C3C' if i == pred_dl_class else '#95A5A6' for i in range(len(class_names))]
    axes[1].barh(label_names, pred_dl_prob * 100, color=colors_bar)
    axes[1].set_title(
        f'MobileNetV2 Fine-Tuning\nPrediksi: {label_names[pred_dl_class]}\n'
        f'Confidence: {pred_dl_prob[pred_dl_class]*100:.1f}%',
        fontsize=10, fontweight='bold'
    )
    axes[1].set_xlabel('Probabilitas (%)')
    axes[1].set_xlim([0, 100])
    axes[1].grid(True, axis='x', alpha=0.3)
    
    # Prediksi CNN + Chi-Square + RF
    colors_bar2 = ['#2ECC71' if i == pred_cnnrf_class else '#95A5A6' for i in range(len(class_names))]
    axes[2].barh(label_names, pred_cnnrf_prob * 100, color=colors_bar2)
    axes[2].set_title(
        f'CNN + Chi² + Random Forest\nPrediksi: {label_names[pred_cnnrf_class]}\n'
        f'Confidence: {pred_cnnrf_prob[pred_cnnrf_class]*100:.1f}%',
        fontsize=10, fontweight='bold'
    )
    axes[2].set_xlabel('Probabilitas (%)')
    axes[2].set_xlim([0, 100])
    axes[2].grid(True, axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('hasil_prediksi.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return pred_dl_class, pred_cnnrf_class


# ============================================================
# DEMO: Prediksi dari sampel test set
# ============================================================

print("Demo prediksi dari sampel test set:")
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Demo Prediksi — MobileNetV2 Fine-Tuning', fontsize=14, fontweight='bold')

# Ambil 8 sampel random dari test set
sample_indices = np.random.choice(len(X_test_dl), 8, replace=False)

for idx, ax in zip(sample_indices, axes.flat):
    img_show = (X_test_dl[idx] + 1) / 2  # Denormalisasi dari [-1,1] ke [0,1]
    img_show = np.clip(img_show, 0, 1)
    
    true_label = class_labels[y_test[idx]]
    pred_label = class_labels[y_pred_dl[idx]]
    is_correct = y_test[idx] == y_pred_dl[idx]
    
    ax.imshow(img_show)
    title_color = 'green' if is_correct else 'red'
    ax.set_title(
        f'Aktual: {true_label}\nPrediksi: {pred_label}\n{"✓ BENAR" if is_correct else "✗ SALAH"}',
        fontsize=8, color=title_color, fontweight='bold'
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('demo_prediksi_sampel.png', dpi=150, bbox_inches='tight')
plt.show()
print("Demo prediksi disimpan: demo_prediksi_sampel.png")

## 10. Simpan Model

In [ ]:
# ============================================================
# SIMPAN MODEL
# ============================================================
import joblib

# Simpan MobileNetV2
mobilenet_model.save('model_mobilenetv2.h5')
print("Model MobileNetV2 disimpan: model_mobilenetv2.h5")

# Simpan Random Forest
joblib.dump(rf_model, 'model_random_forest.pkl')
print("Model Random Forest disimpan: model_random_forest.pkl")

# Simpan CNN+RF (model terbaik)
joblib.dump(rf_cnn_fs, 'model_cnn_rf_best.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(selector, 'feature_selector.pkl')
print("Model CNN+Chi²+RF disimpan: model_cnn_rf_best.pkl")
print("Scaler disimpan: scaler.pkl")
print("Feature selector disimpan: feature_selector.pkl")

print("\n" + "=" * 60)
print("SEMUA MODEL BERHASIL DISIMPAN!")
print("=" * 60)

## 11. Ringkasan & Kesimpulan

In [ ]:
# ============================================================
# RINGKASAN AKHIR
# ============================================================

print("\n" + "=" * 65)
print("RINGKASAN HASIL PENELITIAN")
print("Klasifikasi Bunga: Melati Jakarta, Melati Jepang, Bintaro, Tapak Dara")
print("=" * 65)
print(f"Dataset: {len(X_dl)} gambar | {NUM_CLASSES} kelas | 360 per kelas")
print(f"Train: 80% | Validasi: 10% | Test: 20%")
print(f"Augmentasi: Refleksi, Rotasi ±45°, Scaling 80-120%, Translasi")
print()
print("HASIL PERFORMA MODEL:")
print("-" * 65)
print(f"{'Model':<35} {'Accuracy':>10} {'F1-Score':>10} {'MCC':>8}")
print("-" * 65)
for _, row in all_results.iterrows():
    print(f"{row['Model']:<35} {row['Accuracy (%)']:>9.2f}% {row['F1-Score']:>10.4f} {row['MCC']:>8.4f}")
print("-" * 65)

best_idx  = all_results['Accuracy (%)'].idxmax()
worst_idx = all_results['Accuracy (%)'].idxmin()
print(f"\n✅ Model Terbaik  : {all_results.loc[best_idx, 'Model']}")
print(f"   Accuracy        : {all_results.loc[best_idx, 'Accuracy (%)']}%")
print(f"\n📌 Model Baseline : {all_results.loc[0, 'Model']} (UTS)")
print(f"   Accuracy        : {all_results.loc[0, 'Accuracy (%)']}%")
peningkatan = all_results.loc[best_idx, 'Accuracy (%)'] - all_results.loc[0, 'Accuracy (%)']
print(f"\n📈 Peningkatan Accuracy (DL vs ML): +{peningkatan:.2f}%")
print("=" * 65)